In [0]:
from delta.tables import DeltaTable

# =========================================================================
# 1. CONFIGURACIÓN DE PARÁMETROS DEL PIPELINE INMOBILIARIO
# =========================================================================

dbutils.widgets.text("catalog_param", "")
dbutils.widgets.text("source_schema_param", "")
dbutils.widgets.text("target_schema_param", "")
dbutils.widgets.text("raw_schema_param", "")
# 2. Capturar los valores inyectados por el YAML
catalog_name = dbutils.widgets.get("catalog_param")
source_schema = dbutils.widgets.get("source_schema_param")
target_schema = dbutils.widgets.get("target_schema_param")
raw_schema = dbutils.widgets.get("raw_schema_param")
source_object = "inmuebles" # Cambia esto por el nombre real de tu tabla silver guardada

# CDC Column (La columna de fecha para saber qué datos son nuevos)
cdc_column = "extraction_date"

# Backdated Refresh (Déjalo vacío para carga normal, o pon una fecha para recargar)
backdated_refresh = ""

# Source Fact Table
fact_table = f"{catalog_name}.{source_schema}.{source_object}"

# Target Schema (Capa Gold)
target_object = "fact_inmuebles_mercado"

# Fact Key Cols List (Lo que hace única a una fila en tu tabla de hechos)
# Una misma casa (DimInmuebles) en la misma fecha (extraction_date) no debería tener dos precios.
fact_key_cols = ["DimInmuebleKey", "DimUbicacionKey", "extraction_date"]

# =========================================================================
# 2. DEFINICIÓN DE DIMENSIONES (CRUCE NATURAL -> SUBROGADA)
# =========================================================================

dimensions = [
    {
        "table": f"{catalog_name}.{target_schema}.diminmuebles",
        "alias": "DimInmuebles",
        "sk_column": "DimInmuebleKey", # Nombre de tu llave generada con MD5
        "join_keys": [("Id_inmueble", "Id_inmueble")]  # (Columna_Silver, Columna_Dim)
    },
    {
        "table": f"{catalog_name}.{target_schema}.dimubicaciones",
        "alias": "DimUbicaciones",
        "sk_column": "DimUbicacionKey", # Nombre de tu llave generada con MD5
        "join_keys": [
            ("Estado_Oficial", "Estado_Oficial"),
            ("Municipio_Oficial", "Municipio_Oficial"),
            
        ]
    }
]

# Columns you want to keep from Fact table (Las métricas reales y fechas)
fact_columns = ["Precio", "Area_m2", "Recamaras","Estacionamientos","Banos", "extraction_date"]
# [`f`.`Precio`, `f`.`Area_m2`, `f`.`Fecha`, `f`.`Banos`, `f`.`Pagina`]. SQLSTATE: 42703
# =========================================================================
# 3. LÓGICA DE CARGA INCREMENTAL (WATERMARK)
# =========================================================================

# No Back Dated Refresh
if len(backdated_refresh) == 0:
    # If Table Exists In The Destination
    if spark.catalog.tableExists(f"{catalog_name}.{target_schema}.{target_object}"):
        # Busca la última fecha procesada
        last_load = spark.sql(f"SELECT max({cdc_column}) FROM {catalog_name}.{target_schema}.{target_object}").collect()[0][0]
    else:
        # Carga inicial (Big Bang)
        last_load = "1900-01-01 00:00:00"

# Yes Back Dated Refresh
else:
    last_load = backdated_refresh

print(f"Última fecha de carga detectada: {last_load}")

# =========================================================================
# 4. GENERADOR DINÁMICO DEL QUERY DE HECHOS
# =========================================================================

def generate_fact_query_incremental(fact_table, dimensions, fact_columns, cdc_column, processing_date):
    fact_alias = "f"
    
    # Base columns to select (Las métricas numéricas)
    select_cols = [f"{fact_alias}.{col}" for col in fact_columns]

    # Build joins dynamically
    join_clauses = []
    for dim in dimensions:
        table_full = dim["table"]
        alias = dim["alias"]
        surrogate_key = f"{alias}.{dim['sk_column']}" # Se usa el sk_column explícito
        select_cols.append(surrogate_key)

        # Build ON clause (Cruzando Silver con la Dimensión por las llaves naturales)
        on_conditions = [
            f"{fact_alias}.{fk} = {alias}.{dk}" for fk, dk in dim["join_keys"]
        ]
        join_clause = f"LEFT JOIN {table_full} {alias} ON " + " AND ".join(on_conditions)
        join_clauses.append(join_clause)

    # Final SELECT and JOIN clauses
    select_clause = ",\n    ".join(select_cols)
    joins = "\n".join(join_clauses)

    # WHERE clause for incremental filtering
    where_clause = f"{fact_alias}.{cdc_column} >= '{last_load}'" # Ajustado para fechas de texto/date en Databricks

    # Final query
    query = f"""
SELECT
    {select_clause}
FROM {fact_table} {fact_alias}
{joins}
WHERE {where_clause}
""".strip()

    return query

# =========================================================================
# 5. EJECUCIÓN Y ESCRITURA CON MERGE (UPSERT)
# =========================================================================

query = generate_fact_query_incremental(fact_table, dimensions, fact_columns, cdc_column, last_load)

# Muestra el query generado para validación visual (puedes comentarlo en producción)
print("Query generado para la Tabla de Hechos:\n", query)

df_fact = spark.sql(query)

# Fact Key Columns Merge Condition (Asegura que no haya duplicados si repruebas el mismo día)
fact_key_cols_str = " AND ".join([f"src.{col} = trg.{col}" for col in fact_key_cols])

if spark.catalog.tableExists(f"{catalog_name}.{target_schema}.{target_object}"):
    print("La tabla existe. Ejecutando MERGE incremental...")
    dlt_obj = DeltaTable.forName(spark, f"{catalog_name}.{target_schema}.{target_object}")
    dlt_obj.alias("trg").merge(df_fact.alias("src"), fact_key_cols_str)\
        .whenMatchedUpdateAll(condition = f"src.{cdc_column} >= trg.{cdc_column}")\
        .whenNotMatchedInsertAll()\
        .execute()
else: 
    print("La tabla NO existe. Ejecutando creación inicial (Append)...")
    df_fact.write.format("delta")\
        .mode("append")\
        .saveAsTable(f"{catalog_name}.{target_schema}.{target_object}")

print("¡Proceso de carga a Fact Table finalizado con éxito!")